# DoT Forwarding to Real Upstreams

Variant of the [forwarding lab](dot-forwarding-lab.ipynb) that uses **your real
upstream resolvers** instead of the lab-auth pair (which is *not* started here).
Each server under test runs the `global-forwarder-dot` profile: it listens on
53 + 853 and forwards **everything** over TLS to the upstreams you configure,
validating their certificates against the **system CA bundle**. DNSSEC
validation is enabled in every server profile.

A **NOERROR** rcode counts as PASS — answers are not matched. Instead, every
row surfaces the details that matter:

| column | meaning |
|---|---|
| `flags` | response header flags — **`AD`** present means the answering path validated DNSSEC |
| `do` | DO bit echoed in the response EDNS |
| `edns_payload` | advertised EDNS buffer size |
| `ede` | Extended DNS Errors (RFC 8914), e.g. *why* a SERVFAIL happened |
| `rrsig` | whether RRSIG records came back in the answer |

Note: with real upstreams there is no transport-isolation proof that TLS was
used upstream (that's what the lab-auth marker test in the other notebook is
for) — but every profile here is forward-only with TLS-only upstream
definitions, so there is no plaintext fallback path configured.


## Configuration — edit these


In [ ]:
import dnslab
import pandas as pd

# ---- YOUR UPSTREAMS: real DoT resolvers reachable from the lab ----------
# tls_hostname is used for SNI + certificate validation (system CA bundle).
UPSTREAMS = [
    {'ip': '1.1.1.1', 'port': 853, 'tls_hostname': 'one.one.one.one'},
    {'ip': '9.9.9.9', 'port': 853, 'tls_hostname': 'dns.quad9.net'},
    # {'ip': '10.0.0.53', 'port': 853, 'tls_hostname': 'rec.devries.tv'},
]

# ---- YOUR QUERY NAMES: 'name' (defaults to A) or ('name', 'TYPE') -------
QUERY_NAMES = [
    'example.com',
    ('internetsociety.org', 'A'),      # DNSSEC-signed: expect AD + rrsig
    ('cloudflare.com', 'AAAA'),
    'dnssec-failed.org',               # deliberately bogus: expect SERVFAIL (+EDE)
]

# ---- servers under test (all forward over TLS to UPSTREAMS) --------------
SERVERS = [
    ('unbound',       'global-forwarder-dot'),
    ('bind',          'global-forwarder-dot'),
    ('knot-resolver', 'global-forwarder-dot'),
]

TRANSPORTS = ('do53', 'dot')   # how the notebook queries the servers under test
WANT_DNSSEC = True             # set DO on queries so AD/RRSIG behavior is visible


## 1. Sanity: query your upstreams directly

Confirms each upstream is reachable over TLS from the lab before blaming a
server under test.


In [ ]:
import ssl
import dns.message, dns.query, dns.rcode

for up in UPSTREAMS:
    try:
        q = dns.message.make_query('example.com', 'A')
        r = dns.query.tls(q, up['ip'], port=up.get('port', 853), timeout=5,
                          server_hostname=up['tls_hostname'])
        print(f"ok   {up['ip']:16s} {up['tls_hostname']:24s} rcode={dns.rcode.to_text(r.rcode())}")
    except Exception as e:
        print(f"FAIL {up['ip']:16s} {up['tls_hostname']:24s} {e!r}")


## 2. Start the servers under test

No lab upstreams here — each server gets `UPSTREAMS` rendered into its config.


In [ ]:
for name, profile in SERVERS:
    for inst in dnslab.start(name, profile=profile, upstreams=UPSTREAMS, timeout=120):
        print(f'{inst.name:16s} {inst.profile:22s} {inst.ip}  {inst.status}')
dnslab.status()


## 3. Query matrix

Every name x every server x every transport. PASS = NOERROR.


In [ ]:
targets = dnslab.targets(*[name for name, _ in SERVERS])
report = dnslab.checks.run_query_matrix(targets, QUERY_NAMES,
                                        transports=TRANSPORTS,
                                        want_dnssec=WANT_DNSSEC)
report.style.map(lambda v: {'PASS': 'background-color:#1a7f37;color:white',
                            'FAIL': 'background-color:#cf222e;color:white',
                            'SKIP': 'color:#888'}.get(v, ''), subset=['status'])\
      .map(lambda v: 'font-weight:bold' if isinstance(v, str) and 'AD' in v.split() else '',
           subset=['flags'])


### Compact PASS/FAIL overview


In [ ]:
overview = report.pivot_table(index=['target', 'transport'],
                              columns='qname', values='status',
                              aggfunc='first')
overview


### DNSSEC view

`AD` set and `rrsig=True` on signed names means the path validated and
returned signatures; a SERVFAIL on a known-bogus name means validation is
actually enforced. `ede` (when a server propagates or generates it) tells you why.


In [ ]:
report[['target', 'qname', 'transport', 'rcode', 'flags', 'do', 'ede', 'rrsig', 'latency_ms']]\
    .sort_values(['qname', 'target', 'transport']).reset_index(drop=True)


## 4. Debugging helpers


In [ ]:
# print(dnslab.logs('unbound', tail=50))
# single query, full detail:
# dnslab.checks.query_report(dnslab.target('bind'), 'example.com', 'A', dot=True)


## 5. Teardown


In [ ]:
# dnslab.stop(all=True)
dnslab.status()
